# QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU & TẠO GROUND TRUTH DENSITY MAP
**Mô hình đích:** CSRNet (Dilated Convolutional Neural Networks for Highly Congested Scenes)

---
## 1. Cơ sở Lý thuyết: Tại sao phải dùng Bản đồ Mật độ (Density Map)?
Trong các bài toán đếm đám đông tắc nghẽn (congested scenes):
- Các mô hình Object Detection truyền thống (YOLO, Faster R-CNN) **hoàn toàn thất bại** vì đầu người bị che khuất (occlusion) nghiêm trọng, kích thước đầu người ở hậu cảnh chỉ từ 3-5 pixel, các hộp bounding box sẽ đè khít lên nhau.
- Do đó, nghiên cứu CVPR chuyển bài toán sang dạng **Hồi quy Bản đồ Mật độ (Pixel-wise Density Map Estimation)**:
  - Mỗi điểm đầu người $x_i$ được biểu diễn bằng một hàm xung Dirac $\delta(x - x_i)$.
  - Áp dụng bộ lọc Gaussian 2D làm mờ xung quanh mỗi điểm đầu người:
  $$F(x) = \sum_{i=1}^N \delta(x - x_i) * G_{\sigma_i}(x)$$
  - Khi đó, **tổng tích phân toàn bộ bản đồ mật độ chính là số lượng người trong ảnh**:
  $$N = \int \int F(x, y) \, dx dy \approx \sum_{x, y} F(x, y)$$

## 2. Chiến lược Gaussian: Adaptive vs Fixed Kernel
- **Part A (Đông đúc, mật độ biến thiên mạnh):** Sử dụng **Geometry-adaptive Gaussian Kernel**.
  - Dùng thuật toán **KD-Tree** tìm $k$ người gần nhất ($k=3$). Khoảng cách trung bình $\bar{d}_i$ phản ánh mật độ vùng lân cận (càng đông thì người càng đứng sát nhau $\rightarrow \bar{d}_i$ càng nhỏ).
  - Độ lệch chuẩn thích ứng: $\sigma_i = \beta \cdot \bar{d}_i$ (chọn $\beta = 0.3$).
- **Part B (Thưa thớt, khoảng cách giữa các người lớn):** Sử dụng Gaussian Kernel cố định với $\sigma = 15$.

In [ ]:
import os
import glob
import json
import random
import h5py
import scipy.io as io
import numpy as np
from PIL import Image
import scipy.spatial
import scipy.ndimage
import cv2

# Cố định random seed
random.seed(42)
np.random.seed(42)

WORKSPACE_DIR = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.join(WORKSPACE_DIR, 'Crowd_Counting_ShanghaiTech', 'ShanghaiTech')
print("Workspace Dir:", WORKSPACE_DIR)
print("Dataset Root:", ROOT_DIR)

## 3. Định nghĩa các Hàm Tạo Density Map

In [ ]:
def gaussian_filter_density_adaptive(gt_pts, img_shape):
    """
    Tạo bản đồ mật độ thích ứng hình học (Geometry-adaptive Gaussian) cho Part A
    """
    density = np.zeros(img_shape, dtype=np.float32)
    gt_count = len(gt_pts)
    if gt_count == 0:
        return density

    pts = np.array(gt_pts)
    leafsize = 2048
    tree = scipy.spatial.KDTree(pts.copy(), leafsize=leafsize)
    distances, _ = tree.query(pts, k=min(4, gt_count))

    for i, pt in enumerate(pts):
        x = min(int(round(pt[0])), img_shape[1] - 1)
        y = min(int(round(pt[1])), img_shape[0] - 1)
        
        pt2d = np.zeros(img_shape, dtype=np.float32)
        pt2d[y, x] = 1.0

        if gt_count > 3:
            # beta = 0.3 -> sigma = 0.3 * khoảng cách trung bình đến 3 láng giềng
            sigma = (distances[i][1] + distances[i][2] + distances[i][3]) * 0.1
        else:
            sigma = np.average(np.array(img_shape)) / 4.0

        density += scipy.ndimage.gaussian_filter(pt2d, sigma, mode='constant')

    return density

def gaussian_filter_density_fixed(gt_pts, img_shape, sigma=15.0):
    """
    Tạo bản đồ mật độ Gaussian cố định (Fixed Gaussian) cho Part B
    """
    density = np.zeros(img_shape, dtype=np.float32)
    for pt in gt_pts:
        x = min(int(round(pt[0])), img_shape[1] - 1)
        y = min(int(round(pt[1])), img_shape[0] - 1)
        pt2d = np.zeros(img_shape, dtype=np.float32)
        pt2d[y, x] = 1.0
        density += scipy.ndimage.gaussian_filter(pt2d, sigma, mode='constant')
    return density

## 4. Quy tắc Vàng: Co Tỷ lệ 1/8 và Nhân Hệ số 64
### Tại sao phải co 1/8?
Mạng CSRNet sử dụng frontend gồm **10 tầng tích chập của VGG-16** với **3 tầng MaxPooling** (mỗi tầng có stride=2, kernel=2). Do đó:
$$\text{Output Stride} = 2 \times 2 \times 2 = 8$$
Bản đồ đặc trưng ở đầu ra của mạng sẽ bị thu nhỏ 8 lần theo cả chiều ngang và chiều dọc ($W/8, H/8$).

### Tại sao phải nhân với 64?
Khi thu nhỏ diện tích ảnh đi $8 \times 8 = 64$ lần, tổng diện tích số pixel giảm đi 64 lần. Để **tổng tích phân số người không bị giảm đi 64 lần**, ta bắt buộc phải nhân từng giá trị pixel với 64:
$$\sum \text{DensityMap}_{1/8} = \sum (\text{Resize}_{1/8}(\text{DensityMap}) \times 64) \approx \sum \text{DensityMap}_{\text{gốc}} = N$$

In [ ]:
def process_image(img_path, mat_path, part_name):
    h5_path = img_path.replace('.jpg', '.h5')
    
    # Nếu đã tồn tại file .h5 thì bỏ qua để tiết kiệm thời gian
    if os.path.exists(h5_path):
        return True, 0, 0

    img = Image.open(img_path).convert('RGB')
    w, h = img.size

    mat = io.loadmat(mat_path)
    gt_pts = mat["image_info"][0, 0][0, 0][0]
    actual_count = len(gt_pts)

    if "part_A" in part_name:
        density = gaussian_filter_density_adaptive(gt_pts, (h, w))
    else:
        density = gaussian_filter_density_fixed(gt_pts, (h, w), sigma=15.0)

    # Co 1/8 kích thước và nhân 64
    density_1_8 = cv2.resize(
        density,
        (int(w // 8), int(h // 8)),
        interpolation=cv2.INTER_CUBIC
    ) * 64.0

    with h5py.File(h5_path, 'w') as h5f:
        h5f.create_dataset('density', data=density_1_8)

    computed_count = float(np.sum(density_1_8))
    return False, actual_count, computed_count

def process_dataset_split(base_dir, part_name, split_type):
    split_dir = os.path.join(base_dir, part_name, split_type)
    img_dir = os.path.join(split_dir, 'images')
    gt_dir = os.path.join(split_dir, 'ground-truth')

    img_paths = sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
    print(f"\n--- Đang xử lý {part_name} | {split_type} ({len(img_paths)} ảnh) ---")

    for i, img_path in enumerate(img_paths):
        fname = os.path.splitext(os.path.basename(img_path))[0]
        mat_path = os.path.join(gt_dir, f"GT_{fname}.mat")
        skipped, actual_cnt, comp_cnt = process_image(img_path, mat_path, part_name)
        if (i + 1) % 50 == 0 or (i + 1) == len(img_paths):
            status_str = "Đã có sẵn .h5" if skipped else f"GT={actual_cnt} | H5_Sum={comp_cnt:.2f}"
            print(f"[{i + 1}/{len(img_paths)}] {fname} -> {status_str}")

    # Lưu dưới dạng đường dẫn tương đối (Relative Path) để mọi máy tính clone về đều chạy được
    relative_paths = [os.path.relpath(p, WORKSPACE_DIR).replace('\\', '/') for p in img_paths]
    return relative_paths

## 5. Thực thi Tiền xử lý & Xuất các file JSON Chia tập (90% Train / 10% Val)

In [ ]:
# 1. Xử lý Part A
part_a_train = process_dataset_split(ROOT_DIR, 'part_A', 'train_data')
part_a_test = process_dataset_split(ROOT_DIR, 'part_A', 'test_data')

random.shuffle(part_a_train)
val_len_a = int(len(part_a_train) * 0.1)
a_val_list = part_a_train[:val_len_a]
a_train_list = part_a_train[val_len_a:]

with open('part_A_train.json', 'w') as f:
    json.dump(a_train_list, f, indent=4)
with open('part_A_val.json', 'w') as f:
    json.dump(a_val_list, f, indent=4)
with open('part_A_test.json', 'w') as f:
    json.dump(part_a_test, f, indent=4)
print("\n=> Đã xuất thành công: part_A_train.json, part_A_val.json, part_A_test.json")

# 2. Xử lý Part B
part_b_train = process_dataset_split(ROOT_DIR, 'part_B', 'train_data')
part_b_test = process_dataset_split(ROOT_DIR, 'part_B', 'test_data')

random.shuffle(part_b_train)
val_len_b = int(len(part_b_train) * 0.1)
b_val_list = part_b_train[:val_len_b]
b_train_list = part_b_train[val_len_b:]

with open('part_B_train.json', 'w') as f:
    json.dump(b_train_list, f, indent=4)
with open('part_B_val.json', 'w') as f:
    json.dump(b_val_list, f, indent=4)
with open('part_B_test.json', 'w') as f:
    json.dump(part_b_test, f, indent=4)
print("=> Đã xuất thành công: part_B_train.json, part_B_val.json, part_B_test.json")

## 6. Kiểm tra Chất lượng Nghiệm thu (Sanity Check)
Đảm bảo tổng tích phân bản đồ nhiệt `.h5` sau khi co 1/8 và scale 64 không bị sai lệch quá **0.05 người** so với nhãn `.mat` gốc.

In [ ]:
sample_img = os.path.join(ROOT_DIR, 'part_A/train_data/images/IMG_1.jpg')
sample_mat = os.path.join(ROOT_DIR, 'part_A/train_data/ground-truth/GT_IMG_1.mat')
sample_h5 = sample_img.replace('.jpg', '.h5')

mat = io.loadmat(sample_mat)
gt_count = len(mat["image_info"][0, 0][0, 0][0])

with h5py.File(sample_h5, 'r') as hf:
    density_matrix = np.asarray(hf['density'])
    h5_count = float(np.sum(density_matrix))

error = abs(gt_count - h5_count)
print("=== KIỂM TRA CHẤT LƯỢNG NGHIỆM THU ===")
print(f"Ảnh mẫu: IMG_1.jpg")
print(f"Số người thực tế (Ground Truth Dot Count): {gt_count}")
print(f"Số người tính từ file .h5 (Density Sum) : {h5_count:.4f}")
print(f"Độ sai lệch tuyệt đối                     : {error:.4f}")

if error <= 0.05:
    print("\n-> KẾT QUẢ: ĐẠT TIÊU CHUẨN NGHIỆM THU (Bảo toàn tích phân hoàn hảo)!")
else:
    print("\n-> CẢNH BÁO: Sai lệch vượt quá ngưỡng cho phép, hãy kiểm tra lại hệ số scaling!")